# Dataset B — Feature Selection

Notebook này ghi lại bằng chứng dùng để chốt feature set trước khi huấn luyện mô hình. Luồng kiểm tra bắt đầu từ cohort và target, sau đó đi lần lượt qua các nhóm feature trong `docs/handbook/contracts/feature_governance.md`.

Phạm vi hiện tại: đã hoàn thành data-quality, distribution và redundancy screening cho các nhóm 4.1–4.6. Chưa preprocessing, huấn luyện mô hình, mở test set hoặc thực hiện sensitivity analysis theo protected groups. `construction_method` và `total_units` vẫn chờ representation comparison với `derived_dwelling_category` trước khi khóa final feature set.

In [1]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.utils.path import RAW_STATE_NY_PATH
df_raw = pd.read_csv(RAW_STATE_NY_PATH,dtype='string',keep_default_na=False)

assert ( df_raw.shape[1] == 99)
assert ( df_raw.columns.is_unique)
df_raw.shape

(383614, 99)

## Chuẩn bị cohort và target

Dataset B chỉ giữ các bản ghi có `action_taken` thuộc `{1, 2, 3}`. Target được tạo theo mapping đã chốt:

- `y_approved = 1` khi `action_taken` thuộc `{1, 2}`;
- `y_approved = 0` khi `action_taken = 3`.

`action_taken` được giữ để truy vết và kiểm tra mapping, nhưng không được đưa vào feature matrix `X`.

In [2]:
TARGET_MAPPING = {
    "1": 1,
    "2": 1,
    "3": 0,
}

eligible_mask = df_raw['action_taken'].isin(
    TARGET_MAPPING
)

dataset_b = df_raw.loc[eligible_mask].copy()

dataset_b['y_approved'] = (
    dataset_b['action_taken']
    .map(TARGET_MAPPING)
)

cohort_summary = pd.Series({
    'raw_rows': len(df_raw),
    'dataset_b_rows': len(dataset_b),
    'excluded_row':int((~eligible_mask).sum())
})

cohort_summary

raw_rows          383614
dataset_b_rows    293301
excluded_row       90313
dtype: int64

In [3]:
assert dataset_b['y_approved'].notna().all()
assert set(dataset_b['action_taken'].unique()) == {'1','2','3'}
assert set(dataset_b['y_approved'].unique()) == {0,1}
assert dataset_b.loc[
    dataset_b['action_taken'].isin(['1','2']),
    'y_approved',
].eq(1).all()
assert dataset_b.loc[
    dataset_b["action_taken"].eq("3"),
    "y_approved",
].eq(0).all()

target_check = pd.crosstab(
    dataset_b["action_taken"],
    dataset_b["y_approved"],
    margins=True,
)

target_check

y_approved,0,1,All
action_taken,,,
1,0,208818,208818
2,0,9942,9942
3,74541,0,74541
All,74541,218760,293301


## Feature ưu tiên

### 4.1. Loan Structure and Purpose

Mục tiêu của phần này là kiểm tra ba feature ưu tiên:

- `loan_type`
- `loan_purpose`
- `lien_status`

Các kiểm tra gồm:

1. đối chiếu raw codes với HMDA 2024;
2. đo số lượng và tỷ lệ giá trị hợp lệ;
3. xác định blank hoặc mã ngoài schema;
4. xem phân bố các category trước khi quyết định giữ feature.

Phần này chỉ là data-quality và redundancy screening ban đầu. Chưa thực hiện imputation, encoding, scaling, model training hoặc sensitivity analysis theo protected groups.

In [4]:
GROUP_4_1_COLUMNS = [
    "loan_type",
    "loan_purpose",
    "lien_status",
]

CODE_LABELS_4_1 = {
    "loan_type": {
        "1": "Conventional",
        "2": "FHA",
        "3": "VA",
        "4": "RHS/FSA guaranteed",
    },
    "loan_purpose": {
        "1": "Home purchase",
        "2": "Home improvement",
        "31": "Refinancing",
        "32": "Cash-out refinancing",
        "4": "Other purpose",
        "5": "Not applicable",
    },
    "lien_status": {
        "1": "First lien",
        "2": "Subordinate lien",
    },
}

missing_columns = [
    column
    for column in GROUP_4_1_COLUMNS
    if column not in dataset_b.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

group_4_1_data = dataset_b.loc[:, GROUP_4_1_COLUMNS].copy()

In [5]:
quality_rows = []

for column in GROUP_4_1_COLUMNS:
    valid_codes = set(CODE_LABELS_4_1[column])
    valid_mask = group_4_1_data[column].isin(valid_codes)

    quality_rows.append({
        "feature": column,
        "row_count": len(group_4_1_data),
        "valid_n": int(valid_mask.sum()),
        "invalid_or_missing_n": int((~valid_mask).sum()),
        "valid_pct": round(valid_mask.mean() * 100, 2),
        "raw_unique_count": group_4_1_data[column].nunique(
            dropna=False
        ),
    })

quality_4_1 = pd.DataFrame(quality_rows)

quality_4_1

,feature,row_count,valid_n,invalid_or_missing_n,valid_pct,raw_unique_count
0,loan_type,293301,293301,0,100.0,4
1,loan_purpose,293301,293301,0,100.0,6
2,lien_status,293301,293301,0,100.0,2


In [6]:
def audit_categorical_codes(data, column, code_labels):
    result = (
        data[column]
        .value_counts(dropna=False)
        .rename_axis("raw_code")
        .reset_index(name="n")
    )

    result.insert(0, "feature", column)

    result["share_pct"] = (
        result["n"]
        .div(len(data))
        .mul(100)
        .round(2)
    )

    result["description"] = (
        result["raw_code"]
        .map(code_labels)
        .fillna("Unrecognized or missing")
    )

    result["status"] = (
        result["raw_code"]
        .isin(code_labels)
        .map({
            True: "valid",
            False: "invalid_or_missing",
        })
    )

    return result[
        [
            "feature",
            "raw_code",
            "description",
            "n",
            "share_pct",
            "status",
        ]
    ]

In [7]:
audit_4_1 = pd.concat(
    [
        audit_categorical_codes(
            data=group_4_1_data,
            column=column,
            code_labels=CODE_LABELS_4_1[column],
        )
        for column in GROUP_4_1_COLUMNS
    ],
    ignore_index=True,
)

audit_4_1

,feature,raw_code,description,n,share_pct,status
0,loan_type,1,Conventional,265629,90.57,valid
1,loan_type,2,FHA,21121,7.2,valid
2,loan_type,3,VA,6208,2.12,valid
3,loan_type,4,RHS/FSA guaranteed,343,0.12,valid
4,loan_purpose,1,Home purchase,139144,47.44,valid
5,loan_purpose,2,Home improvement,49762,16.97,valid
6,loan_purpose,4,Other purpose,40360,13.76,valid
7,loan_purpose,32,Cash-out refinancing,34595,11.8,valid
8,loan_purpose,31,Refinancing,29412,10.03,valid
9,loan_purpose,5,Not applicable,28,0.01,valid


In [8]:
invalid_codes_4_1 = audit_4_1.loc[
    audit_4_1["status"] == "invalid_or_missing"
].copy()

invalid_codes_4_1

,feature,raw_code,description,n,share_pct,status


#### Bảng quyết định nhóm 4.1

`decision_4_1` tổng hợp bằng chứng chất lượng dữ liệu, ý nghĩa nghiệp vụ, trạng thái timing, vấn đề redundancy còn chờ và business player dùng cho GSV/SL.

In [9]:
decision_4_1 = quality_4_1[
    [
        'feature',
        'valid_pct',
        'raw_unique_count'
    ]
].copy()

decision_4_1['business_reason'] = (
    decision_4_1['feature'].map({
        "loan_type": (
            "Represents the loan program or guarantee type."
        ),
        "loan_purpose": (
            "Represents the applicant's stated loan purpose."
        ),
        "lien_status": (
            "Represents the proposed lien position."
        ),
    })
)

decision_4_1["timing_status"] = "available_before_decision"

decision_4_1["redundancy_evidence"] = (
    decision_4_1["feature"].map({
        "loan_type": (
            "Review later against derived_loan_product_type."
        ),
        "loan_purpose": (
            "Distinct business concept in group 4.1."
        ),
        "lien_status": (
            "Review later against derived_loan_product_type."
        ),
    })
)

decision_4_1["decision"] = "keep"

decision_4_1["decision_reason"] = (
    decision_4_1["feature"].map({
        "loan_type": (
            "Valid HMDA codes, full coverage and clear "
            "pre-decision business meaning."
        ),
        "loan_purpose": (
            "Valid HMDA codes, full coverage and distinct "
            "business meaning."
        ),
        "lien_status": (
            "Valid HMDA codes, full coverage and clear "
            "collateral-position meaning."
        ),
    })
)

decision_4_1["gsv_player"] = decision_4_1["feature"]

decision_4_1

,feature,valid_pct,raw_unique_count,business_reason,timing_status,redundancy_evidence,decision,decision_reason,gsv_player
0,loan_type,100.0,4,Represents the loan program or guarantee type.,available_before_decision,Review later against derived_loan_product_type.,keep,"Valid HMDA codes, full coverage and clear pre-...",loan_type
1,loan_purpose,100.0,6,Represents the applicant's stated loan purpose.,available_before_decision,Distinct business concept in group 4.1.,keep,"Valid HMDA codes, full coverage and distinct b...",loan_purpose
2,lien_status,100.0,2,Represents the proposed lien position.,available_before_decision,Review later against derived_loan_product_type.,keep,"Valid HMDA codes, full coverage and clear coll...",lien_status


#### Kết luận nhóm 4.1

Cả ba feature `loan_type`, `loan_purpose` và `lien_status` đều có coverage 100% trong cohort Dataset B và không xuất hiện blank hoặc mã ngoài schema HMDA 2024.

`loan_type` chủ yếu là Conventional, trong khi RHS/FSA là category hiếm. `loan_purpose = Not applicable` cũng rất hiếm nhưng vẫn là mã hợp lệ. Các category này được bảo toàn ở giai đoạn feature selection; quyết định xử lý rare category được để lại cho preprocessing sau khi feature set đã khóa.

Ba feature được giữ vì mô tả ba khái niệm nghiệp vụ khác nhau: loại khoản vay, mục đích khoản vay và lien status. Quyết định cuối đối với representation của `loan_type` và `lien_status` sẽ được kiểm tra lại khi so sánh với `derived_loan_product_type`.

Quyết định hiện tại: KEEP cả ba feature.

### 4.2. Transaction Characteristics

Nhóm này kiểm tra ba feature ưu tiên:

- `reverse_mortgage`
- `open-end_line_of_credit`
- `business_or_commercial_purpose`

Ba feature mô tả đặc điểm của giao dịch và không được dùng để lọc
cohort Dataset B.

Mã `1111 = Exempt` là mã HMDA hợp lệ nhưng không cung cấp thông tin
thực về đặc điểm giao dịch. Vì vậy phần này phân biệt schema validity
với informative coverage.

Các kiểm tra gồm:

1. coverage và raw codes;
2. tỷ lệ `Exempt`;
3. mức độ mất cân bằng giữa mã `1` và `2`;
4. co-occurrence giữa ba feature.

Chưa thực hiện target association, protected-group sensitivity,
modeling hoặc sensitivity analysis.

In [10]:
GROUP_4_2_COLUMNS = [
    "reverse_mortgage",
    "open-end_line_of_credit",
    "business_or_commercial_purpose",
]

CODE_LABELS_4_2 = {
    "reverse_mortgage": {
        "1": "Reverse mortgage",
        "2": "Not a reverse mortgage",
        "1111": "Exempt",
    },
    "open-end_line_of_credit": {
        "1": "Open-end line of credit",
        "2": "Not an open-end line of credit",
        "1111": "Exempt",
    },
    "business_or_commercial_purpose": {
        "1": "Primarily business or commercial purpose",
        "2": "Not primarily business or commercial purpose",
        "1111": "Exempt",
    },
}

missing_columns_4_2 = [
    column
    for column in GROUP_4_2_COLUMNS
    if column not in dataset_b.columns
]

assert not missing_columns_4_2, (
    f'Missing columns: {missing_columns_4_2}'
)
group_4_2_data = dataset_b.loc[
    :,
    GROUP_4_2_COLUMNS,
].copy()

In [11]:
quality_rows_4_2 = []

for column in GROUP_4_2_COLUMNS:
    values = group_4_2_data[column]
    valid_codes = set(CODE_LABELS_4_2[column])

    valid_mask = values.isin(valid_codes)
    informative_mask = values.isin(['1','2'])
    exempt_mask = values.eq('1111')
    blank_or_missing_mask = values.fillna('').eq('')

    quality_rows_4_2.append({
        "feature": column,
        "row_count": len(values),
        "valid_n": int(valid_mask.sum()),
        "valid_pct": round(valid_mask.mean() * 100, 2),
        "informative_n": int(informative_mask.sum()),
        "informative_pct": round(
            informative_mask.mean() * 100,
            2,
        ),
        "exempt_n": int(exempt_mask.sum()),
        "exempt_pct": round(exempt_mask.mean() * 100, 2),
        "blank_or_missing_n": int(
            blank_or_missing_mask.sum()
        ),
        "invalid_code_n": int(
            (
                ~valid_mask
                & ~blank_or_missing_mask
            ).sum()
        ),
        "raw_unique_count": values.nunique(
            dropna=False
        ),
    })

quality_4_2 = pd.DataFrame(quality_rows_4_2)

quality_4_2


,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,exempt_n,exempt_pct,blank_or_missing_n,invalid_code_n,raw_unique_count
0,reverse_mortgage,293301,293301,100.0,284665,97.06,8636,2.94,0,0,3
1,open-end_line_of_credit,293301,293301,100.0,284979,97.16,8322,2.84,0,0,3
2,business_or_commercial_purpose,293301,293301,100.0,284680,97.06,8621,2.94,0,0,3


In [12]:
audit_4_2 = pd.concat(
    [
        audit_categorical_codes(
            data=group_4_2_data,
            column=column,
            code_labels=CODE_LABELS_4_2[column],
        )
        for column in GROUP_4_2_COLUMNS
    ],
    ignore_index=True
)

audit_4_2

,feature,raw_code,description,n,share_pct,status
0,reverse_mortgage,2,Not a reverse mortgage,283879,96.79,valid
1,reverse_mortgage,1111,Exempt,8636,2.94,valid
2,reverse_mortgage,1,Reverse mortgage,786,0.27,valid
3,open-end_line_of_credit,2,Not an open-end line of credit,203831,69.5,valid
4,open-end_line_of_credit,1,Open-end line of credit,81148,27.67,valid
5,open-end_line_of_credit,1111,Exempt,8322,2.84,valid
6,business_or_commercial_purpose,2,Not primarily business or commercial purpose,259641,88.52,valid
7,business_or_commercial_purpose,1,Primarily business or commercial purpose,25039,8.54,valid
8,business_or_commercial_purpose,1111,Exempt,8621,2.94,valid


In [13]:
profile_rows_4_2 = []

for column in GROUP_4_2_COLUMNS:
    values = group_4_2_data[column]

    code_counts = {
        "1": int(values.eq("1").sum()),
        "2": int(values.eq("2").sum()),
    }

    informative_n = sum(code_counts.values())

    dominant_code = max(
        code_counts,
        key=code_counts.get,
    )
    minority_code = min(
        code_counts,
        key=code_counts.get,
    )

    if informative_n > 0:
        dominant_pct = round(
            code_counts[dominant_code]
            / informative_n
            * 100,
            2,
        )
        minority_pct = round(
            code_counts[minority_code]
            / informative_n
            * 100,
            2,
        )
    else:
        dominant_pct = None
        minority_pct = None

    profile_rows_4_2.append({
        "feature": column,
        "informative_n": informative_n,
        "code_1_n": code_counts["1"],
        "code_2_n": code_counts["2"],
        "dominant_code": dominant_code,
        "dominant_description": (
            CODE_LABELS_4_2[column][dominant_code]
        ),
        "dominant_pct_informative": dominant_pct,
        "minority_code": minority_code,
        "minority_n": code_counts[minority_code],
        "minority_pct_informative": minority_pct,
    })

profile_4_2 = pd.DataFrame(profile_rows_4_2)

profile_4_2

,feature,informative_n,code_1_n,code_2_n,dominant_code,dominant_description,dominant_pct_informative,minority_code,minority_n,minority_pct_informative
0,reverse_mortgage,284665,786,283879,2,Not a reverse mortgage,99.72,1,786,0.28
1,open-end_line_of_credit,284979,81148,203831,2,Not an open-end line of credit,71.52,1,81148,28.48
2,business_or_commercial_purpose,284680,25039,259641,2,Not primarily business or commercial purpose,91.20,1,25039,8.80


In [14]:
all_informative_mask_4_2 = (
    group_4_2_data
    .isin(['1','2'])
    .all(axis=1)
)
pattern_scope_4_2 = pd.Series({
    "dataset_b_rows": len(group_4_2_data),
    "all_three_informative_n": int(
        all_informative_mask_4_2.sum()
    ),
    "excluded_from_pattern_n": int(
        (~all_informative_mask_4_2).sum()
    ),
    "all_three_informative_pct": round(
        all_informative_mask_4_2.mean() * 100,
        2,
    ),
})

informative_transaction_patterns_4_2 = (
    group_4_2_data.loc[all_informative_mask_4_2]
    .value_counts(dropna=False)
    .rename("n")
    .reset_index()
)

informative_transaction_patterns_4_2["share_pct"] = (
    informative_transaction_patterns_4_2["n"]
    .div(all_informative_mask_4_2.sum())
    .mul(100)
    .round(2)
)

display(pattern_scope_4_2)

informative_transaction_patterns_4_2

dataset_b_rows               293301.00
all_three_informative_n      284656.00
excluded_from_pattern_n        8645.00
all_three_informative_pct        97.05
dtype: float64

,reverse_mortgage,open-end_line_of_credit,business_or_commercial_purpose,n,share_pct
0,2,2,2,179681,63.12
1,2,1,2,79152,27.81
2,2,2,1,23863,8.38
3,2,1,1,1174,0.41
4,1,1,2,642,0.23
5,1,2,2,144,0.05


In [15]:
invalid_codes_4_2 = audit_4_2.loc[
    audit_4_2["status"] == "invalid_or_missing"
].copy()

invalid_codes_4_2

,feature,raw_code,description,n,share_pct,status


In [16]:
decision_4_2 = (
    quality_4_2[
        [
            "feature",
            "valid_pct",
            "informative_pct",
            "exempt_pct",
            "raw_unique_count",
        ]
    ]
    .merge(
        profile_4_2[
            [
                "feature",
                "minority_n",
                "minority_pct_informative",
            ]
        ],
        on="feature",
        validate="one_to_one",
    )
)

decision_4_2["business_reason"] = (
    decision_4_2["feature"].map({
        "reverse_mortgage": (
            "Identifies reverse-mortgage transactions."
        ),
        "open-end_line_of_credit": (
            "Distinguishes open-end from closed-end credit."
        ),
        "business_or_commercial_purpose": (
            "Identifies primarily business-purpose transactions."
        ),
    })
)

decision_4_2["timing_status"] = (
    "available_before_decision"
)

decision_4_2["redundancy_evidence"] = (
    decision_4_2["feature"].map({
        "reverse_mortgage": (
            "Rare but not constant; occurs with both "
            "open-end codes 1 and 2."
        ),
        "open-end_line_of_credit": (
            "Appears across multiple business-purpose patterns."
        ),
        "business_or_commercial_purpose": (
            "Appears with both open-end codes 1 and 2."
        ),
    })
)

decision_4_2["decision"] = "keep"

decision_4_2["decision_reason"] = (
    decision_4_2["feature"].map({
        "reverse_mortgage": (
            "Valid schema, high informative coverage and "
            "a distinct transaction concept despite rarity."
        ),
        "open-end_line_of_credit": (
            "Valid schema, sufficient support and distinct "
            "transaction information."
        ),
        "business_or_commercial_purpose": (
            "Valid schema, sufficient support and distinct "
            "business-purpose information."
        ),
    })
)

decision_4_2["gsv_player"] = (
    "transaction_characteristics"
)

decision_4_2

,feature,valid_pct,informative_pct,exempt_pct,raw_unique_count,minority_n,minority_pct_informative,business_reason,timing_status,redundancy_evidence,decision,decision_reason,gsv_player
0,reverse_mortgage,100.0,97.06,2.94,3,786,0.28,Identifies reverse-mortgage transactions.,available_before_decision,Rare but not constant; occurs with both open-e...,keep,"Valid schema, high informative coverage and a ...",transaction_characteristics
1,open-end_line_of_credit,100.0,97.16,2.84,3,81148,28.48,Distinguishes open-end from closed-end credit.,available_before_decision,Appears across multiple business-purpose patte...,keep,"Valid schema, sufficient support and distinct ...",transaction_characteristics
2,business_or_commercial_purpose,100.0,97.06,2.94,3,25039,8.80,Identifies primarily business-purpose transact...,available_before_decision,Appears with both open-end codes 1 and 2.,keep,"Valid schema, sufficient support and distinct ...",transaction_characteristics


#### Kết luận nhóm 4.2

Cả ba feature `reverse_mortgage`, `open-end_line_of_credit` và
`business_or_commercial_purpose` đều có mã hợp lệ 100%, không xuất
hiện blank, missing hoặc mã ngoài schema. Informative coverage của
cả ba feature đều trên 97%; phần còn lại chủ yếu là mã
`1111 = Exempt`.

`reverse_mortgage` là feature hiếm với 786 hồ sơ mã `1`, tương đương
0.28% số dòng informative. Feature này gần constant nhưng vẫn được
giữ vì mô tả một loại giao dịch riêng và không được suy ra hoàn toàn
từ `open-end_line_of_credit`.

`open-end_line_of_credit` và `business_or_commercial_purpose` có số
lượng category thiểu số đủ lớn. Các transaction patterns cho thấy ba
feature không lặp lại nhau theo quan hệ xác định, vì vậy không cần
thực hiện thêm Cramér’s V hoặc sensitivity analysis ở giai đoạn này.

Quyết định hiện tại: KEEP cả ba raw feature trong model. Khi tính
GSV/SL, ba feature được gom thành một business player
`transaction_characteristics`. Mã `1111 = Exempt` được bảo toàn như
một category riêng trong preprocessing, không chuyển thành mã `2`.

### 4.3. Financial Capacity

Nhóm này kiểm tra hai feature ưu tiên:

- `income`
- `debt_to_income_ratio`

`income` mô tả quy mô thu nhập được sử dụng trong quá trình ra quyết
định, trong khi `debt_to_income_ratio` mô tả gánh nặng nghĩa vụ nợ
tương đối so với thu nhập. Hai feature có liên quan về mặt nghiệp vụ
nhưng không được coi là các representation thay thế nhau.

Các kiểm tra gồm:

1. coverage và khả năng parse `income`;
2. các trạng thái `NA`, `Exempt`, blank và token không hợp lệ;
3. phân bố của giá trị income đã parse;
4. domain và sparsity của các DTI bands;
5. phân bố income trong từng DTI band để kiểm tra overlap.

Chưa thực hiện imputation, clipping, log transformation, encoding,
target association, modeling hoặc sensitivity analysis.

In [17]:
GROUP_4_3_COLUMNS = [
    'income',
    'debt_to_income_ratio'
]

DTI_INFORMATIVE_VALUES_4_3 = [
    "<20%",
    "20%-<30%",
    "30%-<36%",
    *[str(value) for value in range(36, 50)],
    "50%-60%",
    ">60%",
]

DTI_SPECIAL_VALUES_4_3 = [
    "NA",
    "Exempt",
]

DTI_VALID_VALUES_4_3 = (
    DTI_INFORMATIVE_VALUES_4_3
    + DTI_SPECIAL_VALUES_4_3
)

missing_columns_4_3 = [
    column
    for column in GROUP_4_3_COLUMNS
    if column not in dataset_b.columns
]

assert not missing_columns_4_3, (
    f"Missing columns: {missing_columns_4_3}"
)

group_4_3_data = dataset_b.loc[
    :,
    GROUP_4_3_COLUMNS,
].copy()

In [18]:
income_raw_4_3 = group_4_3_data["income"]

income_numeric_4_3 = pd.to_numeric(
    income_raw_4_3,
    errors="coerce",
)

income_parseable_mask_4_3 = (
    income_numeric_4_3.notna()
)

income_na_mask_4_3 = income_raw_4_3.eq("NA")

income_blank_or_missing_mask_4_3 = (
    income_raw_4_3
    .fillna("")
    .eq("")
)

income_invalid_mask_4_3 = ~(
    income_parseable_mask_4_3
    | income_na_mask_4_3
    | income_blank_or_missing_mask_4_3
)

income_quality_row_4_3 = {
    "feature": "income",
    "row_count": len(income_raw_4_3),
    "valid_n": int(
        (
            income_parseable_mask_4_3
            | income_na_mask_4_3
        ).sum()
    ),
    "valid_pct": round(
        (
            income_parseable_mask_4_3
            | income_na_mask_4_3
        ).mean() * 100,
        2,
    ),
    "informative_n": int(
        income_parseable_mask_4_3.sum()
    ),
    "informative_pct": round(
        income_parseable_mask_4_3.mean() * 100,
        2,
    ),
    "special_n": int(income_na_mask_4_3.sum()),
    "special_pct": round(
        income_na_mask_4_3.mean() * 100,
        2,
    ),
    "blank_or_missing_n": int(
        income_blank_or_missing_mask_4_3.sum()
    ),
    "invalid_token_n": int(
        income_invalid_mask_4_3.sum()
    ),
    "raw_unique_count": income_raw_4_3.nunique(
        dropna=False
    ),
}

In [19]:
dti_raw_4_3 = group_4_3_data[
    "debt_to_income_ratio"
]

dti_informative_mask_4_3 = dti_raw_4_3.isin(
    DTI_INFORMATIVE_VALUES_4_3
)

dti_special_mask_4_3 = dti_raw_4_3.isin(
    DTI_SPECIAL_VALUES_4_3
)

dti_blank_or_missing_mask_4_3 = (
    dti_raw_4_3
    .fillna("")
    .eq("")
)

dti_valid_mask_4_3 = (
    dti_informative_mask_4_3
    | dti_special_mask_4_3
)

dti_invalid_mask_4_3 = ~(
    dti_valid_mask_4_3
    | dti_blank_or_missing_mask_4_3
)

dti_quality_row_4_3 = {
    "feature": "debt_to_income_ratio",
    "row_count": len(dti_raw_4_3),
    "valid_n": int(dti_valid_mask_4_3.sum()),
    "valid_pct": round(
        dti_valid_mask_4_3.mean() * 100,
        2,
    ),
    "informative_n": int(
        dti_informative_mask_4_3.sum()
    ),
    "informative_pct": round(
        dti_informative_mask_4_3.mean() * 100,
        2,
    ),
    "special_n": int(dti_special_mask_4_3.sum()),
    "special_pct": round(
        dti_special_mask_4_3.mean() * 100,
        2,
    ),
    "blank_or_missing_n": int(
        dti_blank_or_missing_mask_4_3.sum()
    ),
    "invalid_token_n": int(
        dti_invalid_mask_4_3.sum()
    ),
    "raw_unique_count": dti_raw_4_3.nunique(
        dropna=False
    ),
}

quality_4_3 = pd.DataFrame([
    income_quality_row_4_3,
    dti_quality_row_4_3,
])

quality_4_3

,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,special_n,special_pct,blank_or_missing_n,invalid_token_n,raw_unique_count
0,income,293301,293301,100.0,272723,92.98,20578,7.02,0,0,3007
1,debt_to_income_ratio,293301,293301,100.0,262193,89.39,31108,10.61,0,0,21


In [20]:
income_observed_4_3 = income_numeric_4_3.loc[
    income_parseable_mask_4_3
]

income_distribution_4_3 = pd.Series({
    "numeric_n": len(income_observed_4_3),
    "negative_n": int(
        income_observed_4_3.lt(0).sum()
    ),
    "zero_n": int(
        income_observed_4_3.eq(0).sum()
    ),
    "positive_n": int(
        income_observed_4_3.gt(0).sum()
    ),
    "minimum": income_observed_4_3.min(),
    "p01": income_observed_4_3.quantile(0.01),
    "q1": income_observed_4_3.quantile(0.25),
    "median": income_observed_4_3.median(),
    "q3": income_observed_4_3.quantile(0.75),
    "p99": income_observed_4_3.quantile(0.99),
    "maximum": income_observed_4_3.max(),
})

income_distribution_4_3

numeric_n        272723.0
negative_n          360.0
zero_n             3111.0
positive_n       269252.0
minimum          -11229.0
p01                   0.0
q1                   75.0
median              120.0
q3                  190.0
p99                1242.0
maximum       100000000.0
dtype: float64

In [21]:
dti_status_map_4_3 = {
    **{
        value: "informative"
        for value in DTI_INFORMATIVE_VALUES_4_3
    },
    **{
        value: "special"
        for value in DTI_SPECIAL_VALUES_4_3
    },
}

dti_display_order_4_3 = {
    value: position
    for position, value in enumerate(
        DTI_VALID_VALUES_4_3
    )
}

audit_dti_4_3 = (
    dti_raw_4_3
    .value_counts(dropna=False)
    .rename_axis("raw_value")
    .reset_index(name="n")
)

audit_dti_4_3["share_pct"] = (
    audit_dti_4_3["n"]
    .div(len(dti_raw_4_3))
    .mul(100)
    .round(2)
)

audit_dti_4_3["status"] = (
    audit_dti_4_3["raw_value"]
    .map(dti_status_map_4_3)
    .fillna("invalid_or_missing")
)

audit_dti_4_3["display_order"] = (
    audit_dti_4_3["raw_value"]
    .map(dti_display_order_4_3)
    .fillna(len(DTI_VALID_VALUES_4_3))
)

audit_dti_4_3 = (
    audit_dti_4_3
    .sort_values("display_order")
    .drop(columns="display_order")
    .reset_index(drop=True)
)

audit_dti_4_3

,raw_value,n,share_pct,status
0,<20%,15077,5.14,informative
1,20%-<30%,36366,12.4,informative
2,30%-<36%,36082,12.3,informative
3,36,6910,2.36,informative
4,37,7359,2.51,informative
5,38,7751,2.64,informative
6,39,8295,2.83,informative
7,40,8412,2.87,informative
8,41,8943,3.05,informative
9,42,9999,3.41,informative


In [22]:
invalid_dti_tokens_4_3 = audit_dti_4_3.loc[
    audit_dti_4_3["status"]
    == "invalid_or_missing"
].copy()

invalid_dti_tokens_4_3

,raw_value,n,share_pct,status


In [23]:
joint_informative_mask_4_3 = (
    income_parseable_mask_4_3
    & dti_informative_mask_4_3
)

financial_overlap_data_4_3 = pd.DataFrame({
    "income_numeric": income_numeric_4_3.loc[
        joint_informative_mask_4_3
    ],
    "dti_band": dti_raw_4_3.loc[
        joint_informative_mask_4_3
    ],
})

financial_overlap_data_4_3["dti_band"] = (
    pd.Categorical(
        financial_overlap_data_4_3["dti_band"],
        categories=DTI_INFORMATIVE_VALUES_4_3,
        ordered=True,
    )
)

overlap_scope_4_3 = pd.Series({
    "dataset_b_rows": len(group_4_3_data),
    "both_informative_n": int(
        joint_informative_mask_4_3.sum()
    ),
    "both_informative_pct": round(
        joint_informative_mask_4_3.mean() * 100,
        2,
    ),
})

income_by_dti_4_3 = (
    financial_overlap_data_4_3
    .groupby(
        "dti_band",
        observed=True,
        sort=True,
    )
    .agg(
        n=("income_numeric", "size"),
        income_q1=(
            "income_numeric",
            lambda values: values.quantile(0.25),
        ),
        income_median=("income_numeric", "median"),
        income_q3=(
            "income_numeric",
            lambda values: values.quantile(0.75),
        ),
    )
    .reset_index()
)

income_by_dti_4_3["income_iqr"] = (
    income_by_dti_4_3["income_q3"]
    - income_by_dti_4_3["income_q1"]
)

display(overlap_scope_4_3)

income_by_dti_4_3

dataset_b_rows          293301.00
both_informative_n      259767.00
both_informative_pct        88.57
dtype: float64

,dti_band,n,income_q1,income_median,income_q3,income_iqr
0,<20%,14557,100,196.0,445.00,345.0
1,20%-<30%,35934,100,160.0,265.00,165.0
2,30%-<36%,35746,89,140.0,220.00,131.0
3,36,6847,84,132.0,207.00,123.0
4,37,7293,82,129.0,200.00,118.0
5,38,7687,80,127.0,195.00,115.0
6,39,8220,82,128.0,196.25,114.25
7,40,8324,80,126.0,190.00,110.0
8,41,8868,79,123.0,189.00,110.0
9,42,9920,79,124.0,188.00,109.0


#### Kiểm tra nhất quán nhóm 4.3

In [24]:
assert len(group_4_3_data) == len(dataset_b)
assert quality_4_3["row_count"].eq(len(dataset_b)).all()
assert quality_4_3["valid_n"].eq(len(dataset_b)).all()
assert quality_4_3["blank_or_missing_n"].eq(0).all()
assert quality_4_3["invalid_token_n"].eq(0).all()

assert (
    int(income_distribution_4_3["negative_n"])
    + int(income_distribution_4_3["zero_n"])
    + int(income_distribution_4_3["positive_n"])
    == int(income_distribution_4_3["numeric_n"])
)

assert int(audit_dti_4_3["n"].sum()) == len(dataset_b)
assert invalid_dti_tokens_4_3.empty
assert int(income_by_dti_4_3["n"].sum()) == int(
    joint_informative_mask_4_3.sum()
)
assert income_by_dti_4_3["income_iqr"].gt(0).all()

validation_4_3 = pd.Series({
    "row_count_reconciled": True,
    "valid_domain_reconciled": True,
    "income_counts_reconciled": True,
    "dti_counts_reconciled": True,
    "overlap_counts_reconciled": True,
    "income_varies_within_every_dti_band": True,
})

validation_4_3

row_count_reconciled                   True
valid_domain_reconciled                True
income_counts_reconciled               True
dti_counts_reconciled                  True
overlap_counts_reconciled              True
income_varies_within_every_dti_band    True
dtype: bool

#### Bảng quyết định nhóm 4.3

`decision_4_3` tổng hợp chất lượng dữ liệu, đặc điểm phân phối, bằng chứng overlap và business player dùng cho GSV/SL.

In [25]:
decision_4_3 = quality_4_3[
    [
        "feature",
        "valid_pct",
        "informative_pct",
        "special_pct",
        "raw_unique_count",
    ]
].copy()

decision_4_3["business_reason"] = (
    decision_4_3["feature"].map({
        "income": (
            "Represents the scale of reported applicant income."
        ),
        "debt_to_income_ratio": (
            "Represents debt burden relative to income."
        ),
    })
)

decision_4_3["timing_status"] = (
    "available_before_decision"
)

decision_4_3["distribution_evidence"] = (
    decision_4_3["feature"].map({
        "income": (
            f"{int(income_distribution_4_3['negative_n'])} negative, "
            f"{int(income_distribution_4_3['zero_n'])} zero; "
            "heavy upper tail retained for later preprocessing."
        ),
        "debt_to_income_ratio": (
            f"All {len(income_by_dti_4_3)} informative DTI bands "
            "are represented."
        ),
    })
)

decision_4_3["redundancy_evidence"] = (
    decision_4_3["feature"].map({
        "income": (
            "Income has positive IQR within every DTI band; "
            "it is not determined by DTI."
        ),
        "debt_to_income_ratio": (
            "DTI retains relative debt-burden information "
            "that income alone does not provide."
        ),
    })
)

decision_4_3["decision"] = "keep"

decision_4_3["decision_reason"] = (
    decision_4_3["feature"].map({
        "income": (
            "Valid domain, high informative coverage and a "
            "distinct financial-capacity concept."
        ),
        "debt_to_income_ratio": (
            "Valid domain, adequate coverage and distinct "
            "relative debt-burden information."
        ),
    })
)

decision_4_3["gsv_player"] = decision_4_3["feature"]

decision_4_3

,feature,valid_pct,informative_pct,special_pct,raw_unique_count,business_reason,timing_status,distribution_evidence,redundancy_evidence,decision,decision_reason,gsv_player
0,income,100.0,92.98,7.02,3007,Represents the scale of reported applicant inc...,available_before_decision,"360 negative, 3111 zero; heavy upper tail reta...",Income has positive IQR within every DTI band;...,keep,"Valid domain, high informative coverage and a ...",income
1,debt_to_income_ratio,100.0,89.39,10.61,21,Represents debt burden relative to income.,available_before_decision,All 19 informative DTI bands are represented.,DTI retains relative debt-burden information t...,keep,"Valid domain, adequate coverage and distinct r...",debt_to_income_ratio


#### Kết luận nhóm 4.3

Cả `income` và `debt_to_income_ratio` đều có domain hợp lệ 100%, không xuất hiện blank, missing hoặc token ngoài schema. `income` có 272.723 giá trị numeric, tương đương 92,98% cohort; 20.578 dòng còn lại là `NA`. DTI có 262.193 giá trị informative, tương đương 89,39%; 31.108 dòng còn lại là `NA` hoặc `Exempt`.

Phân phối `income` có 360 giá trị âm, 3.111 giá trị bằng 0 và đuôi phải rất dài: median là 120, p99 là 1.242 nhưng maximum đạt 100.000.000 theo raw unit. Đây là vấn đề cần xử lý có kiểm soát trong preprocessing; các giá trị không bị loại, clip hoặc log-transform ở giai đoạn feature selection.

Tất cả 19 DTI bands đều có quan sát và không có category hiếm đến mức mất support. Trên 259.767 hồ sơ có đồng thời income và DTI informative, income vẫn có IQR dương trong mọi DTI band. Median income nhìn chung giảm khi DTI tăng nhưng các khoảng phân phối vẫn chồng lấn đáng kể. Vì vậy hai feature có association nhưng không phải representation thay thế nhau.

Quyết định hiện tại: KEEP cả `income` và `debt_to_income_ratio` như hai raw feature riêng trong model và hai business players riêng cho GSV/SL. Không cần sensitivity analysis bổ sung ở nhóm 4.3.

### 4.4. Loan and Collateral Scale

Nhóm này kiểm tra ba feature ưu tiên:

- `loan_amount`
- `property_value`
- `loan_to_value_ratio`

`loan_amount` và `property_value` mô tả quy mô khoản vay và tài sản bảo đảm, trong khi `loan_to_value_ratio` mô tả mức đòn bẩy tương đối. Đây là block redundancy quan trọng vì LTV có quan hệ gần toán học với hai feature còn lại.

Các kiểm tra gồm:

1. coverage, khả năng parse và special token;
2. phân phối bền vững của các giá trị numeric;
3. common population cho từng cặp và cả ba feature;
4. độ nhất quán giữa LTV báo cáo và tỷ lệ `loan_amount / property_value`;
5. Spearman correlation trên cùng common population.

Tỷ lệ tự tính chỉ là diagnostic để đánh giá consistency, không phải feature mới. Chưa thực hiện imputation, clipping, transformation, target association, modeling hoặc sensitivity analysis.

In [26]:
GROUP_4_4_COLUMNS = [
    "loan_amount",
    "property_value",
    "loan_to_value_ratio",
]

SPECIAL_VALUES_4_4 = {
    "loan_amount": [],
    "property_value": ["NA", "Exempt"],
    "loan_to_value_ratio": ["NA", "Exempt"],
}

missing_columns_4_4 = [
    column
    for column in GROUP_4_4_COLUMNS
    if column not in dataset_b.columns
]

assert not missing_columns_4_4, (
    f"Missing columns: {missing_columns_4_4}"
)

group_4_4_data = dataset_b.loc[
    :,
    GROUP_4_4_COLUMNS,
].copy()

numeric_data_4_4 = group_4_4_data.apply(
    lambda values: pd.to_numeric(
        values,
        errors="coerce",
    )
)

In [27]:
quality_rows_4_4 = []

for column in GROUP_4_4_COLUMNS:
    raw_values = group_4_4_data[column]
    numeric_mask = numeric_data_4_4[column].notna()
    special_mask = raw_values.isin(
        SPECIAL_VALUES_4_4[column]
    )
    blank_or_missing_mask = (
        raw_values
        .fillna("")
        .eq("")
    )
    invalid_mask = ~(
        numeric_mask
        | special_mask
        | blank_or_missing_mask
    )

    quality_rows_4_4.append({
        "feature": column,
        "row_count": len(raw_values),
        "valid_n": int(
            (numeric_mask | special_mask).sum()
        ),
        "valid_pct": round(
            (numeric_mask | special_mask).mean() * 100,
            2,
        ),
        "numeric_n": int(numeric_mask.sum()),
        "numeric_pct": round(
            numeric_mask.mean() * 100,
            2,
        ),
        "special_n": int(special_mask.sum()),
        "special_pct": round(
            special_mask.mean() * 100,
            2,
        ),
        "blank_or_missing_n": int(
            blank_or_missing_mask.sum()
        ),
        "invalid_token_n": int(
            invalid_mask.sum()
        ),
        "raw_unique_count": raw_values.nunique(
            dropna=False
        ),
    })

quality_4_4 = pd.DataFrame(quality_rows_4_4)

quality_4_4

,feature,row_count,valid_n,valid_pct,numeric_n,numeric_pct,special_n,special_pct,blank_or_missing_n,invalid_token_n,raw_unique_count
0,loan_amount,293301,293301,100.0,293301,100.00,0,0.00,0,0,993
1,property_value,293301,293301,100.0,279550,95.31,13751,4.69,0,0,1223
2,loan_to_value_ratio,293301,293301,100.0,274681,93.65,18620,6.35,0,0,80236


In [28]:
distribution_rows_4_4 = []

for column in GROUP_4_4_COLUMNS:
    observed_values = (
        numeric_data_4_4[column]
        .dropna()
    )

    distribution_rows_4_4.append({
        "feature": column,
        "numeric_n": len(observed_values),
        "negative_n": int(
            observed_values.lt(0).sum()
        ),
        "zero_n": int(
            observed_values.eq(0).sum()
        ),
        "positive_n": int(
            observed_values.gt(0).sum()
        ),
        "minimum": observed_values.min(),
        "p01": observed_values.quantile(0.01),
        "q1": observed_values.quantile(0.25),
        "median": observed_values.median(),
        "q3": observed_values.quantile(0.75),
        "p99": observed_values.quantile(0.99),
        "maximum": observed_values.max(),
    })

distribution_4_4 = pd.DataFrame(
    distribution_rows_4_4
)

distribution_4_4

,feature,numeric_n,negative_n,zero_n,positive_n,minimum,p01,q1,median,q3,p99,maximum
0,loan_amount,293301,0,0,293301,5000.000,15000.000,105000.000,205000.00,435000.000,2435000.000,6.008550e+08
1,property_value,279550,0,0,279550,5000.000,75000.000,255000.000,495000.00,775000.000,4445000.000,1.400005e+09
2,loan_to_value_ratio,274681,0,0,274681,0.001,8.235,55.285,74.99,84.444,105.021,2.000000e+06


In [29]:
ltv_numeric_4_4 = (
    numeric_data_4_4["loan_to_value_ratio"]
    .dropna()
)

ltv_range_profile_4_4 = pd.Series({
    "numeric_n": len(ltv_numeric_4_4),
    "above_100_n": int(
        ltv_numeric_4_4.gt(100).sum()
    ),
    "above_100_pct": round(
        ltv_numeric_4_4.gt(100).mean() * 100,
        2,
    ),
    "above_200_n": int(
        ltv_numeric_4_4.gt(200).sum()
    ),
    "above_200_pct": round(
        ltv_numeric_4_4.gt(200).mean() * 100,
        4,
    ),
    "above_1000_n": int(
        ltv_numeric_4_4.gt(1000).sum()
    ),
    "above_1000_pct": round(
        ltv_numeric_4_4.gt(1000).mean() * 100,
        4,
    ),
    "p99_9": ltv_numeric_4_4.quantile(0.999),
    "maximum": ltv_numeric_4_4.max(),
})

ltv_range_profile_4_4

numeric_n         2.746810e+05
above_100_n       5.261000e+03
above_100_pct     1.920000e+00
above_200_n       2.460000e+02
above_200_pct     8.960000e-02
above_1000_n      6.000000e+01
above_1000_pct    2.180000e-02
p99_9             1.934654e+02
maximum           2.000000e+06
dtype: float64

In [30]:
special_token_rows_4_4 = []
invalid_token_frames_4_4 = []

for column in GROUP_4_4_COLUMNS:
    raw_values = group_4_4_data[column]
    numeric_mask = numeric_data_4_4[column].notna()
    special_mask = raw_values.isin(
        SPECIAL_VALUES_4_4[column]
    )
    blank_or_missing_mask = (
        raw_values
        .fillna("")
        .eq("")
    )
    invalid_mask = ~(
        numeric_mask
        | special_mask
        | blank_or_missing_mask
    )

    for token in SPECIAL_VALUES_4_4[column]:
        token_n = int(raw_values.eq(token).sum())
        special_token_rows_4_4.append({
            "feature": column,
            "raw_value": token,
            "n": token_n,
            "share_pct": round(
                token_n / len(raw_values) * 100,
                2,
            ),
            "status": "special",
        })

    blank_n = int(blank_or_missing_mask.sum())
    special_token_rows_4_4.append({
        "feature": column,
        "raw_value": "<blank_or_missing>",
        "n": blank_n,
        "share_pct": round(
            blank_n / len(raw_values) * 100,
            2,
        ),
        "status": "missing",
    })

    invalid_counts = (
        raw_values.loc[invalid_mask]
        .value_counts(dropna=False)
        .rename_axis("raw_value")
        .reset_index(name="n")
    )
    invalid_counts.insert(0, "feature", column)
    invalid_token_frames_4_4.append(invalid_counts)

special_tokens_4_4 = pd.DataFrame(
    special_token_rows_4_4
)

invalid_tokens_4_4 = pd.concat(
    invalid_token_frames_4_4,
    ignore_index=True,
)

display(special_tokens_4_4)

invalid_tokens_4_4

,feature,raw_value,n,share_pct,status
0,loan_amount,<blank_or_missing>,0,0.00,missing
1,property_value,NA,5109,1.74,special
2,property_value,Exempt,8642,2.95,special
3,property_value,<blank_or_missing>,0,0.00,missing
4,loan_to_value_ratio,NA,9978,3.40,special
5,loan_to_value_ratio,Exempt,8642,2.95,special
6,loan_to_value_ratio,<blank_or_missing>,0,0.00,missing


,feature,raw_value,n


In [31]:
numeric_masks_4_4 = numeric_data_4_4.notna()

overlap_definitions_4_4 = {
    "loan_amount_numeric": (
        numeric_masks_4_4["loan_amount"]
    ),
    "property_value_numeric": (
        numeric_masks_4_4["property_value"]
    ),
    "loan_to_value_ratio_numeric": (
        numeric_masks_4_4["loan_to_value_ratio"]
    ),
    "loan_amount_and_property_value": (
        numeric_masks_4_4["loan_amount"]
        & numeric_masks_4_4["property_value"]
    ),
    "loan_amount_and_ltv": (
        numeric_masks_4_4["loan_amount"]
        & numeric_masks_4_4["loan_to_value_ratio"]
    ),
    "property_value_and_ltv": (
        numeric_masks_4_4["property_value"]
        & numeric_masks_4_4["loan_to_value_ratio"]
    ),
    "all_three_numeric": (
        numeric_masks_4_4.all(axis=1)
    ),
}

overlap_scope_4_4 = pd.DataFrame([
    {
        "scope": scope,
        "n": int(mask.sum()),
        "share_pct": round(mask.mean() * 100, 2),
    }
    for scope, mask in overlap_definitions_4_4.items()
])

overlap_scope_4_4

,scope,n,share_pct
0,loan_amount_numeric,293301,100.00
1,property_value_numeric,279550,95.31
2,loan_to_value_ratio_numeric,274681,93.65
3,loan_amount_and_property_value,279550,95.31
4,loan_amount_and_ltv,274681,93.65
5,property_value_and_ltv,273800,93.35
6,all_three_numeric,273800,93.35


In [32]:
all_three_numeric_mask_4_4 = (
    numeric_masks_4_4.all(axis=1)
)

ltv_comparison_mask_4_4 = (
    all_three_numeric_mask_4_4
    & numeric_data_4_4["property_value"].gt(0)
)

ltv_consistency_data_4_4 = (
    numeric_data_4_4.loc[
        ltv_comparison_mask_4_4,
        GROUP_4_4_COLUMNS,
    ]
    .copy()
)

ltv_consistency_data_4_4["calculated_ltv"] = (
    ltv_consistency_data_4_4["loan_amount"]
    .div(ltv_consistency_data_4_4["property_value"])
    .mul(100)
)

ltv_consistency_data_4_4["absolute_difference"] = (
    ltv_consistency_data_4_4["loan_to_value_ratio"]
    .sub(ltv_consistency_data_4_4["calculated_ltv"])
    .abs()
)

absolute_difference_4_4 = (
    ltv_consistency_data_4_4["absolute_difference"]
)

ltv_consistency_4_4 = pd.Series({
    "all_three_numeric_n": int(
        all_three_numeric_mask_4_4.sum()
    ),
    "comparison_n": len(ltv_consistency_data_4_4),
    "comparison_pct": round(
        ltv_comparison_mask_4_4.mean() * 100,
        2,
    ),
    "within_1_point_pct": round(
        absolute_difference_4_4.le(1).mean() * 100,
        2,
    ),
    "within_5_points_pct": round(
        absolute_difference_4_4.le(5).mean() * 100,
        2,
    ),
    "median_absolute_difference": (
        absolute_difference_4_4.median()
    ),
    "p95_absolute_difference": (
        absolute_difference_4_4.quantile(0.95)
    ),
    "maximum_absolute_difference": (
        absolute_difference_4_4.max()
    ),
})

ltv_consistency_4_4

all_three_numeric_n            2.738000e+05
comparison_n                   2.738000e+05
comparison_pct                 9.335000e+01
within_1_point_pct             4.525000e+01
within_5_points_pct            6.663000e+01
median_absolute_difference     1.285385e+00
p95_absolute_difference        6.481707e+01
maximum_absolute_difference    1.999700e+06
dtype: float64

In [33]:
common_numeric_data_4_4 = (
    numeric_data_4_4.loc[
        all_three_numeric_mask_4_4,
        GROUP_4_4_COLUMNS,
    ]
    .copy()
)

association_4_4 = common_numeric_data_4_4.corr(
    method="spearman"
).round(4)

association_4_4

,loan_amount,property_value,loan_to_value_ratio
loan_amount,1.0000,0.6080,0.2377
property_value,0.6080,1.0000,-0.2526
loan_to_value_ratio,0.2377,-0.2526,1.0000


#### Kiểm tra nhất quán nhóm 4.4

In [34]:
assert len(group_4_4_data) == len(dataset_b)
assert quality_4_4["row_count"].eq(
    len(dataset_b)
).all()

reconciled_counts_4_4 = (
    quality_4_4["numeric_n"]
    + quality_4_4["special_n"]
    + quality_4_4["blank_or_missing_n"]
    + quality_4_4["invalid_token_n"]
)
assert reconciled_counts_4_4.eq(
    len(dataset_b)
).all()
assert quality_4_4["blank_or_missing_n"].eq(0).all()
assert quality_4_4["invalid_token_n"].eq(0).all()
assert invalid_tokens_4_4.empty

assert distribution_4_4["numeric_n"].equals(
    quality_4_4["numeric_n"]
)
assert (
    distribution_4_4["negative_n"]
    + distribution_4_4["zero_n"]
    + distribution_4_4["positive_n"]
).equals(distribution_4_4["numeric_n"])

expected_common_n_4_4 = int(
    all_three_numeric_mask_4_4.sum()
)
observed_common_n_4_4 = int(
    overlap_scope_4_4.loc[
        overlap_scope_4_4["scope"]
        == "all_three_numeric",
        "n",
    ].iloc[0]
)
assert observed_common_n_4_4 == expected_common_n_4_4
assert len(common_numeric_data_4_4) == expected_common_n_4_4
assert len(ltv_consistency_data_4_4) <= expected_common_n_4_4
assert association_4_4.index.tolist() == GROUP_4_4_COLUMNS
assert association_4_4.columns.tolist() == GROUP_4_4_COLUMNS

validation_4_4 = pd.Series({
    "row_count_reconciled": True,
    "domain_counts_reconciled": True,
    "no_blank_or_invalid_tokens": True,
    "distribution_counts_reconciled": True,
    "common_population_reconciled": True,
    "ltv_comparison_scope_reconciled": True,
    "association_scope_reconciled": True,
})

validation_4_4

row_count_reconciled               True
domain_counts_reconciled           True
no_blank_or_invalid_tokens         True
distribution_counts_reconciled     True
common_population_reconciled       True
ltv_comparison_scope_reconciled    True
association_scope_reconciled       True
dtype: bool

#### Bảng quyết định nhóm 4.4

`decision_4_4` tổng hợp chất lượng dữ liệu, bằng chứng consistency, mức độ redundancy và business player dùng cho GSV/SL.

In [35]:
decision_4_4 = quality_4_4[
    [
        "feature",
        "valid_pct",
        "numeric_pct",
        "special_pct",
        "raw_unique_count",
    ]
].copy()

decision_4_4["business_reason"] = (
    decision_4_4["feature"].map({
        "loan_amount": (
            "Represents the scale of requested financing."
        ),
        "property_value": (
            "Represents the reported collateral scale."
        ),
        "loan_to_value_ratio": (
            "Represents leverage relative to property value."
        ),
    })
)

decision_4_4["timing_status"] = (
    "available_before_decision"
)

quality_lookup_4_4 = quality_4_4.set_index("feature")

decision_4_4["quality_evidence"] = (
    decision_4_4["feature"].map({
        "loan_amount": (
            f"{quality_lookup_4_4.loc['loan_amount', 'numeric_pct']:.2f}% "
            "numeric coverage; no blank or invalid token."
        ),
        "property_value": (
            f"{quality_lookup_4_4.loc['property_value', 'numeric_pct']:.2f}% "
            "numeric coverage; "
            f"{quality_lookup_4_4.loc['property_value', 'special_pct']:.2f}% "
            "NA or Exempt."
        ),
        "loan_to_value_ratio": (
            f"{quality_lookup_4_4.loc['loan_to_value_ratio', 'numeric_pct']:.2f}% "
            "numeric coverage; "
            f"{quality_lookup_4_4.loc['loan_to_value_ratio', 'special_pct']:.2f}% "
            "NA or Exempt; "
            f"{int(ltv_range_profile_4_4['above_1000_n'])} "
            "numeric values exceed 1000."
        ),
    })
)

decision_4_4["redundancy_evidence"] = (
    decision_4_4["feature"].map({
        "loan_amount": (
            "Retained as the financing-scale anchor."
        ),
        "property_value": (
            "Shares scale information with loan amount and is the "
            "denominator concept represented by reported LTV."
        ),
        "loan_to_value_ratio": (
            "Reported LTV is not an exact arithmetic duplicate: "
            f"{ltv_consistency_4_4['within_5_points_pct']:.2f}% "
            "are within 5 points of calculated LTV."
        ),
    })
)

decision_4_4["decision"] = (
    decision_4_4["feature"].map({
        "loan_amount": "keep",
        "property_value": "exclude_from_main",
        "loan_to_value_ratio": "keep",
    })
)

decision_4_4["decision_reason"] = (
    decision_4_4["feature"].map({
        "loan_amount": (
            "Complete coverage and a necessary absolute-scale concept."
        ),
        "property_value": (
            "Excluded to avoid retaining all three related "
            "representations in the main feature set."
        ),
        "loan_to_value_ratio": (
            "Adequate coverage and distinct relative-leverage meaning; "
            "extreme values require train-fitted preprocessing."
        ),
    })
)

decision_4_4["gsv_player"] = (
    decision_4_4["feature"].map({
        "loan_amount": "loan_amount",
        "property_value": pd.NA,
        "loan_to_value_ratio": "loan_to_value_ratio",
    })
)

decision_4_4

,feature,valid_pct,numeric_pct,special_pct,raw_unique_count,business_reason,timing_status,quality_evidence,redundancy_evidence,decision,decision_reason,gsv_player
0,loan_amount,100.0,100.00,0.00,993,Represents the scale of requested financing.,available_before_decision,100.00% numeric coverage; no blank or invalid ...,Retained as the financing-scale anchor.,keep,Complete coverage and a necessary absolute-sca...,loan_amount
1,property_value,100.0,95.31,4.69,1223,Represents the reported collateral scale.,available_before_decision,95.31% numeric coverage; 4.69% NA or Exempt.,Shares scale information with loan amount and ...,exclude_from_main,Excluded to avoid retaining all three related ...,<NA>
2,loan_to_value_ratio,100.0,93.65,6.35,80236,Represents leverage relative to property value.,available_before_decision,93.65% numeric coverage; 6.35% NA or Exempt; 6...,Reported LTV is not an exact arithmetic duplic...,keep,Adequate coverage and distinct relative-levera...,loan_to_value_ratio


#### Kết luận nhóm 4.4

Cả ba feature đều có domain hợp lệ 100% và không xuất hiện blank hoặc token ngoài schema. `NA` và `Exempt` được giữ như special values hợp lệ, không được coi là giá trị numeric. `loan_amount` có numeric coverage 100%; `property_value` có 279.550 giá trị numeric (95,31%) và 13.751 dòng `NA`/`Exempt`; `loan_to_value_ratio` có 274.681 giá trị numeric (93,65%) và 18.620 dòng `NA`/`Exempt`. Common population có đồng thời cả ba giá trị numeric gồm 273.800 hồ sơ, tương đương 93,35% cohort.

Trên common population, Spearman correlation là 0,6080 giữa `loan_amount` và `property_value`, 0,2377 giữa `loan_amount` và LTV, và -0,2526 giữa `property_value` và LTV. Đây là bằng chứng association, không phải feature importance hoặc quan hệ nhân quả. LTV báo cáo cũng không phải bản sao số học chính xác của `loan_amount / property_value`: median absolute difference là 1,285 điểm và 66,63% hồ sơ nằm trong phạm vi 5 điểm. Sự khác biệt này được giữ như bằng chứng về representation, không được tự động gán cho một nguyên nhân cụ thể.

LTV có đuôi cực trị cần xử lý ở preprocessing: 60 giá trị numeric lớn hơn 1.000, tương đương 0,0218%, và maximum là 2.000.000. Các giá trị này chưa bị xóa, clip hoặc thay đổi trong feature selection. Việc giữ LTV không đồng nghĩa coi mọi giá trị cực trị là hợp lý.

Quyết định hiện tại: KEEP `loan_amount` và `loan_to_value_ratio` như hai raw feature riêng trong model và hai business players riêng cho GSV/SL. EXCLUDE `property_value` khỏi main feature set để không giữ đồng thời cả ba representation có liên hệ; cột vẫn được giữ nguyên trong dữ liệu nguồn để truy vết. Bằng chứng business, coverage và redundancy đã đủ để chốt nhóm 4.4 nên chưa cần model sensitivity, target association hoặc protected-group analysis.

### 4.5. Loan Term

Nhóm này kiểm tra feature ưu tiên `loan_term`, đại diện cho thời hạn cam kết của khoản vay. Giá trị được giữ theo raw unit của source; phần này không tự chuyển đổi đơn vị hoặc tạo term bands.

Các kiểm tra gồm:

1. coverage và khả năng parse thành số;
2. các trạng thái `NA`, `Exempt`, blank và token không hợp lệ;
3. phân phối bền vững của các giá trị numeric;
4. các thời hạn xuất hiện phổ biến;
5. range diagnostics cho các giá trị rất ngắn, rất dài hoặc không chia hết cho 12.

Các range diagnostics chỉ mô tả hình dạng dữ liệu, không tự động gắn nhãn giá trị là invalid. Chưa thực hiện imputation, clipping, scaling, encoding, target association, modeling hoặc sensitivity analysis.

In [36]:
GROUP_4_5_COLUMNS = ["loan_term"]
LOAN_TERM_SPECIAL_VALUES_4_5 = [
    "NA",
    "Exempt",
]

missing_columns_4_5 = [
    column
    for column in GROUP_4_5_COLUMNS
    if column not in dataset_b.columns
]

assert not missing_columns_4_5, (
    f"Missing columns: {missing_columns_4_5}"
)

group_4_5_data = dataset_b.loc[
    :,
    GROUP_4_5_COLUMNS,
].copy()

loan_term_raw_4_5 = group_4_5_data["loan_term"]
loan_term_numeric_4_5 = pd.to_numeric(
    loan_term_raw_4_5,
    errors="coerce",
)

In [37]:
loan_term_numeric_mask_4_5 = (
    loan_term_numeric_4_5.notna()
)
loan_term_special_mask_4_5 = (
    loan_term_raw_4_5.isin(
        LOAN_TERM_SPECIAL_VALUES_4_5
    )
)
loan_term_blank_or_missing_mask_4_5 = (
    loan_term_raw_4_5
    .fillna("")
    .eq("")
)
loan_term_invalid_mask_4_5 = ~(
    loan_term_numeric_mask_4_5
    | loan_term_special_mask_4_5
    | loan_term_blank_or_missing_mask_4_5
)

quality_4_5 = pd.DataFrame([{
    "feature": "loan_term",
    "row_count": len(loan_term_raw_4_5),
    "valid_n": int(
        (
            loan_term_numeric_mask_4_5
            | loan_term_special_mask_4_5
        ).sum()
    ),
    "valid_pct": round(
        (
            loan_term_numeric_mask_4_5
            | loan_term_special_mask_4_5
        ).mean() * 100,
        2,
    ),
    "numeric_n": int(
        loan_term_numeric_mask_4_5.sum()
    ),
    "numeric_pct": round(
        loan_term_numeric_mask_4_5.mean() * 100,
        2,
    ),
    "special_n": int(
        loan_term_special_mask_4_5.sum()
    ),
    "special_pct": round(
        loan_term_special_mask_4_5.mean() * 100,
        2,
    ),
    "blank_or_missing_n": int(
        loan_term_blank_or_missing_mask_4_5.sum()
    ),
    "invalid_token_n": int(
        loan_term_invalid_mask_4_5.sum()
    ),
    "raw_unique_count": loan_term_raw_4_5.nunique(
        dropna=False
    ),
}])

quality_4_5

,feature,row_count,valid_n,valid_pct,numeric_n,numeric_pct,special_n,special_pct,blank_or_missing_n,invalid_token_n,raw_unique_count
0,loan_term,293301,293301,100.0,274648,93.64,18653,6.36,0,0,338


In [38]:
loan_term_observed_4_5 = (
    loan_term_numeric_4_5.loc[
        loan_term_numeric_mask_4_5
    ]
)

distribution_4_5 = pd.Series({
    "numeric_n": len(loan_term_observed_4_5),
    "negative_n": int(
        loan_term_observed_4_5.lt(0).sum()
    ),
    "zero_n": int(
        loan_term_observed_4_5.eq(0).sum()
    ),
    "positive_n": int(
        loan_term_observed_4_5.gt(0).sum()
    ),
    "minimum": loan_term_observed_4_5.min(),
    "p01": loan_term_observed_4_5.quantile(0.01),
    "q1": loan_term_observed_4_5.quantile(0.25),
    "median": loan_term_observed_4_5.median(),
    "q3": loan_term_observed_4_5.quantile(0.75),
    "p99": loan_term_observed_4_5.quantile(0.99),
    "maximum": loan_term_observed_4_5.max(),
    "non_integer_n": int(
        loan_term_observed_4_5.mod(1).ne(0).sum()
    ),
})

distribution_4_5

numeric_n        274648.0
negative_n            0.0
zero_n                0.0
positive_n       274648.0
minimum               1.0
p01                  12.0
q1                  300.0
median              360.0
q3                  360.0
p99                 360.0
maximum             900.0
non_integer_n         0.0
dtype: float64

In [39]:
loan_term_frequency_all_4_5 = (
    loan_term_observed_4_5
    .value_counts()
    .rename_axis("loan_term")
    .reset_index(name="n")
)

loan_term_frequency_all_4_5["share_numeric_pct"] = (
    loan_term_frequency_all_4_5["n"]
    .div(len(loan_term_observed_4_5))
    .mul(100)
    .round(2)
)

loan_term_frequency_4_5 = (
    loan_term_frequency_all_4_5
    .head(15)
    .copy()
)

loan_term_frequency_4_5

,loan_term,n,share_numeric_pct
0,360,189844,69.12
1,240,27838,10.14
2,300,20394,7.43
3,180,14051,5.12
4,120,9589,3.49
5,12,2724,0.99
6,60,2678,0.98
7,480,1695,0.62
8,84,784,0.29
9,372,680,0.25


In [40]:
special_token_rows_4_5 = []

for token in LOAN_TERM_SPECIAL_VALUES_4_5:
    token_n = int(loan_term_raw_4_5.eq(token).sum())
    special_token_rows_4_5.append({
        "raw_value": token,
        "n": token_n,
        "share_pct": round(
            token_n / len(loan_term_raw_4_5) * 100,
            2,
        ),
        "status": "special",
    })

blank_n_4_5 = int(
    loan_term_blank_or_missing_mask_4_5.sum()
)
special_token_rows_4_5.append({
    "raw_value": "<blank_or_missing>",
    "n": blank_n_4_5,
    "share_pct": round(
        blank_n_4_5 / len(loan_term_raw_4_5) * 100,
        2,
    ),
    "status": "missing",
})

special_tokens_4_5 = pd.DataFrame(
    special_token_rows_4_5
)

invalid_tokens_4_5 = (
    loan_term_raw_4_5.loc[
        loan_term_invalid_mask_4_5
    ]
    .value_counts(dropna=False)
    .rename_axis("raw_value")
    .reset_index(name="n")
)

display(special_tokens_4_5)

invalid_tokens_4_5

,raw_value,n,share_pct,status
0,NA,10360,3.53,special
1,Exempt,8293,2.83,special
2,<blank_or_missing>,0,0.00,missing


,raw_value,n


In [41]:
range_profile_4_5 = pd.Series({
    "numeric_n": len(loan_term_observed_4_5),
    "below_12_n": int(
        loan_term_observed_4_5.lt(12).sum()
    ),
    "below_12_pct": round(
        loan_term_observed_4_5.lt(12).mean() * 100,
        4,
    ),
    "above_360_n": int(
        loan_term_observed_4_5.gt(360).sum()
    ),
    "above_360_pct": round(
        loan_term_observed_4_5.gt(360).mean() * 100,
        4,
    ),
    "above_480_n": int(
        loan_term_observed_4_5.gt(480).sum()
    ),
    "above_480_pct": round(
        loan_term_observed_4_5.gt(480).mean() * 100,
        4,
    ),
    "above_600_n": int(
        loan_term_observed_4_5.gt(600).sum()
    ),
    "above_600_pct": round(
        loan_term_observed_4_5.gt(600).mean() * 100,
        4,
    ),
    "not_multiple_12_n": int(
        loan_term_observed_4_5.mod(12).ne(0).sum()
    ),
    "not_multiple_12_pct": round(
        loan_term_observed_4_5.mod(12).ne(0).mean()
        * 100,
        4,
    ),
})

range_profile_4_5

numeric_n              274648.0000
below_12_n                 78.0000
below_12_pct                0.0284
above_360_n              2702.0000
above_360_pct               0.9838
above_480_n                 9.0000
above_480_pct               0.0033
above_600_n                 1.0000
above_600_pct               0.0004
not_multiple_12_n        1953.0000
not_multiple_12_pct         0.7111
dtype: float64

#### Kiểm tra nhất quán nhóm 4.5

In [42]:
assert len(group_4_5_data) == len(dataset_b)
assert int(quality_4_5.loc[0, "row_count"]) == len(
    dataset_b
)

reconciled_count_4_5 = int(
    quality_4_5.loc[0, "numeric_n"]
    + quality_4_5.loc[0, "special_n"]
    + quality_4_5.loc[0, "blank_or_missing_n"]
    + quality_4_5.loc[0, "invalid_token_n"]
)
assert reconciled_count_4_5 == len(dataset_b)
assert int(
    quality_4_5.loc[0, "blank_or_missing_n"]
) == 0
assert int(
    quality_4_5.loc[0, "invalid_token_n"]
) == 0
assert invalid_tokens_4_5.empty

assert (
    int(distribution_4_5["negative_n"])
    + int(distribution_4_5["zero_n"])
    + int(distribution_4_5["positive_n"])
    == int(distribution_4_5["numeric_n"])
)
assert int(distribution_4_5["negative_n"]) == 0
assert int(distribution_4_5["zero_n"]) == 0
assert int(distribution_4_5["non_integer_n"]) == 0
assert int(
    loan_term_frequency_all_4_5["n"].sum()
) == int(distribution_4_5["numeric_n"])
assert int(special_tokens_4_5["n"].sum()) == int(
    quality_4_5.loc[0, "special_n"]
    + quality_4_5.loc[0, "blank_or_missing_n"]
)

validation_4_5 = pd.Series({
    "row_count_reconciled": True,
    "domain_counts_reconciled": True,
    "no_blank_or_invalid_tokens": True,
    "numeric_counts_reconciled": True,
    "all_numeric_values_positive": True,
    "all_numeric_values_integer": True,
    "frequency_counts_reconciled": True,
    "special_counts_reconciled": True,
})

validation_4_5

row_count_reconciled           True
domain_counts_reconciled       True
no_blank_or_invalid_tokens     True
numeric_counts_reconciled      True
all_numeric_values_positive    True
all_numeric_values_integer     True
frequency_counts_reconciled    True
special_counts_reconciled      True
dtype: bool

#### Bảng quyết định nhóm 4.5

`decision_4_5` tổng hợp chất lượng dữ liệu, đặc điểm phân phối, range diagnostics và business player dùng cho GSV/SL.

In [43]:
decision_4_5 = quality_4_5[
    [
        "feature",
        "valid_pct",
        "numeric_pct",
        "special_pct",
        "raw_unique_count",
    ]
].copy()

decision_4_5["business_reason"] = (
    "Represents the contracted duration of the loan."
)
decision_4_5["timing_status"] = (
    "available_before_decision"
)
decision_4_5["quality_evidence"] = (
    f"{quality_4_5.loc[0, 'numeric_pct']:.2f}% numeric; "
    f"{quality_4_5.loc[0, 'special_pct']:.2f}% NA or Exempt; "
    "no blank or invalid token."
)
decision_4_5["distribution_evidence"] = (
    f"Median and p99 are {distribution_4_5['median']:.0f} "
    f"raw units; maximum is {distribution_4_5['maximum']:.0f}."
)
decision_4_5["range_evidence"] = (
    f"{int(range_profile_4_5['above_360_n'])} values exceed 360; "
    f"{int(range_profile_4_5['above_480_n'])} exceed 480; "
    "diagnostic only, not an invalid-code rule."
)
decision_4_5["decision"] = "keep"
decision_4_5["decision_reason"] = (
    "Valid domain, high numeric coverage and a distinct "
    "contract-duration concept; extreme values remain for "
    "train-fitted preprocessing review."
)
decision_4_5["gsv_player"] = "loan_term"

decision_4_5

,feature,valid_pct,numeric_pct,special_pct,raw_unique_count,business_reason,timing_status,quality_evidence,distribution_evidence,range_evidence,decision,decision_reason,gsv_player
0,loan_term,100.0,93.64,6.36,338,Represents the contracted duration of the loan.,available_before_decision,93.64% numeric; 6.36% NA or Exempt; no blank o...,Median and p99 are 360 raw units; maximum is 900.,2702 values exceed 360; 9 exceed 480; diagnost...,keep,"Valid domain, high numeric coverage and a dist...",loan_term


#### Kết luận nhóm 4.5

`loan_term` có domain hợp lệ 100% và không xuất hiện blank hoặc token ngoài schema. Có 274.648 giá trị numeric, tương đương 93,64% cohort; 18.653 dòng còn lại là special values hợp lệ, gồm 10.360 `NA` (3,53%) và 8.293 `Exempt` (2,83%).

Toàn bộ giá trị numeric là số nguyên dương. Minimum là 1, Q1 là 300, median là 360, Q3 là 360, p99 là 360 và maximum là 900 theo raw unit của source. Giá trị 360 chiếm 69,12% numeric observations; các giá trị phổ biến tiếp theo là 240 (10,14%), 300 (7,43%) và 180 (5,12%). Phân phối vì vậy tập trung mạnh ở một số thời hạn phổ biến nhưng vẫn có các giá trị khác được giữ nguyên.

Range diagnostics ghi nhận 78 giá trị nhỏ hơn 12 (0,0284%), 2.702 giá trị lớn hơn 360 (0,9838%), 9 giá trị lớn hơn 480 (0,0033%), 1 giá trị lớn hơn 600 (0,0004%) và 1.953 giá trị không chia hết cho 12 (0,7111%). Đây là các nhóm cần được nhận diện trong preprocessing review, không phải bằng chứng rằng chúng không hợp lệ. Phần này không tự chuyển đổi raw unit, tạo term bands, xóa hoặc clip các giá trị này.

Quyết định hiện tại: KEEP `loan_term` như một raw feature riêng trong model và một business player riêng cho GSV/SL. Feature có coverage đủ cao, domain sạch và biểu diễn khái niệm thời hạn hợp đồng khác với quy mô khoản vay, LTV hoặc năng lực tài chính. Không cần redundancy analysis, model sensitivity, target association hoặc protected-group analysis ở nhóm 4.5.

### 4.6. Property Characteristics

Nhóm này kiểm tra ba feature ưu tiên:

- `occupancy_type`
- `construction_method`
- `total_units`

Cả ba feature được audit như categorical values theo schema HMDA. Đặc biệt, `total_units` gồm cả mã đơn vị riêng lẻ và các khoảng như `5-24`, `25-49`, `50-99`, `100-149`, `>149`; cột này không được parse thành số liên tục. Code labels được đối chiếu với [HMDA 2024 Filing Instructions Guide](https://files.ffiec.cfpb.gov/documentation/2024-hmda-fig.pdf) và [HMDA Data Browser](https://ffiec.cfpb.gov/data-browser/maps/2024).

Các kiểm tra gồm:

1. valid codes, blank và token ngoài schema;
2. coverage, dominant category và minority support;
3. bảng joint pattern giữa `construction_method` và `total_units`;
4. trạng thái representation so với `derived_dwelling_category`.

Chưa tạo derived feature, chưa gộp category, chưa thực hiện target association, protected-group analysis, modeling hoặc sensitivity analysis.

In [44]:
GROUP_4_6_COLUMNS = [
    "occupancy_type",
    "construction_method",
    "total_units",
]

TOTAL_UNITS_ORDER_4_6 = [
    "1",
    "2",
    "3",
    "4",
    "5-24",
    "25-49",
    "50-99",
    "100-149",
    ">149",
]

CODE_LABELS_4_6 = {
    "occupancy_type": {
        "1": "Principal residence",
        "2": "Second residence",
        "3": "Investment property",
    },
    "construction_method": {
        "1": "Site-built",
        "2": "Manufactured home",
    },
    "total_units": {
        value: (
            f"{value} unit"
            if value == "1"
            else f"{value} units"
        )
        for value in TOTAL_UNITS_ORDER_4_6
    },
}

missing_columns_4_6 = [
    column
    for column in GROUP_4_6_COLUMNS
    if column not in dataset_b.columns
]

assert not missing_columns_4_6, (
    f"Missing columns: {missing_columns_4_6}"
)

group_4_6_data = dataset_b.loc[
    :,
    GROUP_4_6_COLUMNS,
].copy()

In [45]:
quality_rows_4_6 = []

for column in GROUP_4_6_COLUMNS:
    values = group_4_6_data[column]
    valid_codes = set(CODE_LABELS_4_6[column])
    valid_mask = values.isin(valid_codes)
    blank_or_missing_mask = (
        values
        .fillna("")
        .eq("")
    )
    invalid_mask = ~(
        valid_mask
        | blank_or_missing_mask
    )

    valid_counts = values.loc[valid_mask].value_counts()

    quality_rows_4_6.append({
        "feature": column,
        "row_count": len(values),
        "valid_n": int(valid_mask.sum()),
        "valid_pct": round(
            valid_mask.mean() * 100,
            2,
        ),
        "blank_or_missing_n": int(
            blank_or_missing_mask.sum()
        ),
        "invalid_code_n": int(invalid_mask.sum()),
        "raw_unique_count": values.nunique(
            dropna=False
        ),
        "dominant_n": int(valid_counts.max()),
        "dominant_pct": round(
            valid_counts.max() / len(values) * 100,
            2,
        ),
        "minority_n": int(valid_counts.min()),
        "minority_pct": round(
            valid_counts.min() / len(values) * 100,
            4,
        ),
    })

quality_4_6 = pd.DataFrame(quality_rows_4_6)

quality_4_6

,feature,row_count,valid_n,valid_pct,blank_or_missing_n,invalid_code_n,raw_unique_count,dominant_n,dominant_pct,minority_n,minority_pct
0,occupancy_type,293301,293301,100.0,0,0,3,257433,87.77,7100,2.4207
1,construction_method,293301,293301,100.0,0,0,2,286486,97.68,6815,2.3236
2,total_units,293301,293301,100.0,0,0,9,259784,88.57,108,0.0368


In [46]:
audit_4_6 = pd.concat(
    [
        audit_categorical_codes(
            data=group_4_6_data,
            column=column,
            code_labels=CODE_LABELS_4_6[column],
        )
        for column in GROUP_4_6_COLUMNS
    ],
    ignore_index=True,
)

audit_4_6

,feature,raw_code,description,n,share_pct,status
0,occupancy_type,1,Principal residence,257433,87.77,valid
1,occupancy_type,3,Investment property,28768,9.81,valid
2,occupancy_type,2,Second residence,7100,2.42,valid
3,construction_method,1,Site-built,286486,97.68,valid
4,construction_method,2,Manufactured home,6815,2.32,valid
5,total_units,1,1 unit,259784,88.57,valid
6,total_units,2,2 units,23905,8.15,valid
7,total_units,3,3 units,4534,1.55,valid
8,total_units,5-24,5-24 units,2091,0.71,valid
9,total_units,4,4 units,1971,0.67,valid


In [47]:
invalid_codes_4_6 = audit_4_6.loc[
    audit_4_6["status"] == "invalid_or_missing"
].copy()

profile_rows_4_6 = []

for column in GROUP_4_6_COLUMNS:
    feature_audit = audit_4_6.loc[
        (audit_4_6["feature"] == column)
        & (audit_4_6["status"] == "valid")
    ].sort_values("n", ascending=False)

    dominant_row = feature_audit.iloc[0]
    minority_row = feature_audit.iloc[-1]

    profile_rows_4_6.append({
        "feature": column,
        "dominant_code": dominant_row["raw_code"],
        "dominant_n": int(dominant_row["n"]),
        "dominant_pct": dominant_row["share_pct"],
        "minority_code": minority_row["raw_code"],
        "minority_n": int(minority_row["n"]),
        "minority_pct": minority_row["share_pct"],
    })

profile_4_6 = pd.DataFrame(profile_rows_4_6)

display(invalid_codes_4_6)

profile_4_6

,feature,raw_code,description,n,share_pct,status


,feature,dominant_code,dominant_n,dominant_pct,minority_code,minority_n,minority_pct
0,occupancy_type,1,257433,87.77,2,7100,2.42
1,construction_method,1,286486,97.68,2,6815,2.32
2,total_units,1,259784,88.57,100-149,108,0.04


In [48]:
construction_units_patterns_4_6 = (
    pd.crosstab(
        group_4_6_data["construction_method"],
        group_4_6_data["total_units"],
    )
    .reindex(
        index=["1", "2"],
        columns=TOTAL_UNITS_ORDER_4_6,
        fill_value=0,
    )
    .rename(
        index=CODE_LABELS_4_6["construction_method"]
    )
)

joint_scope_4_6 = pd.Series({
    "possible_combinations": (
        len(CODE_LABELS_4_6["construction_method"])
        * len(CODE_LABELS_4_6["total_units"])
    ),
    "observed_combinations": int(
        construction_units_patterns_4_6.gt(0).sum().sum()
    ),
    "smallest_observed_cell_n": int(
        construction_units_patterns_4_6
        .where(construction_units_patterns_4_6.gt(0))
        .min()
        .min()
    ),
    "largest_observed_cell_n": int(
        construction_units_patterns_4_6.max().max()
    ),
})

display(joint_scope_4_6)

construction_units_patterns_4_6

possible_combinations           18
observed_combinations           18
smallest_observed_cell_n         2
largest_observed_cell_n     253039
dtype: int64

total_units,1,2,3,4,5-24,25-49,50-99,100-149,>149
construction_method,,,,,,,,,
Site-built,253039,23881,4528,1968,2073,398,328,103,168
Manufactured home,6745,24,6,3,18,7,5,5,2


#### Kiểm tra nhất quán nhóm 4.6

In [49]:
assert len(group_4_6_data) == len(dataset_b)
assert quality_4_6["row_count"].eq(
    len(dataset_b)
).all()
assert quality_4_6["valid_n"].eq(
    len(dataset_b)
).all()
assert quality_4_6["blank_or_missing_n"].eq(0).all()
assert quality_4_6["invalid_code_n"].eq(0).all()
assert invalid_codes_4_6.empty

audit_totals_4_6 = (
    audit_4_6
    .groupby("feature")["n"]
    .sum()
)
assert audit_totals_4_6.eq(len(dataset_b)).all()

expected_unique_counts_4_6 = pd.Series({
    column: len(CODE_LABELS_4_6[column])
    for column in GROUP_4_6_COLUMNS
})
observed_unique_counts_4_6 = (
    quality_4_6
    .set_index("feature")["raw_unique_count"]
)
assert observed_unique_counts_4_6.equals(
    expected_unique_counts_4_6
)

assert int(
    construction_units_patterns_4_6.to_numpy().sum()
) == len(dataset_b)
assert int(joint_scope_4_6["observed_combinations"]) == int(
    joint_scope_4_6["possible_combinations"]
)

validation_4_6 = pd.Series({
    "row_count_reconciled": True,
    "valid_domain_reconciled": True,
    "no_blank_or_invalid_codes": True,
    "all_schema_categories_observed": True,
    "audit_counts_reconciled": True,
    "joint_counts_reconciled": True,
    "all_construction_unit_combinations_observed": True,
})

validation_4_6

row_count_reconciled                           True
valid_domain_reconciled                        True
no_blank_or_invalid_codes                      True
all_schema_categories_observed                 True
audit_counts_reconciled                        True
joint_counts_reconciled                        True
all_construction_unit_combinations_observed    True
dtype: bool

#### Bảng quyết định nhóm 4.6

`decision_4_6` tách data-quality decision khỏi final representation decision. `construction_method` và `total_units` chỉ được chuyển tiếp tới comparison với `derived_dwelling_category`; chúng chưa được khóa thành hai GSV/SL players riêng.

In [50]:
decision_4_6 = quality_4_6[
    [
        "feature",
        "valid_pct",
        "raw_unique_count",
        "dominant_pct",
        "minority_n",
        "minority_pct",
    ]
].copy()

decision_4_6["business_reason"] = (
    decision_4_6["feature"].map({
        "occupancy_type": (
            "Represents how the property will be occupied."
        ),
        "construction_method": (
            "Distinguishes site-built from manufactured housing."
        ),
        "total_units": (
            "Represents dwelling-unit count as HMDA categories."
        ),
    })
)

decision_4_6["timing_status"] = (
    "available_before_decision"
)

decision_4_6["quality_evidence"] = (
    decision_4_6["feature"].map({
        feature: (
            f"{quality_4_6.set_index('feature').loc[feature, 'valid_pct']:.2f}% "
            "valid; no blank or invalid code."
        )
        for feature in GROUP_4_6_COLUMNS
    })
)

decision_4_6["sparsity_evidence"] = (
    decision_4_6["feature"].map({
        "occupancy_type": (
            f"Smallest category has "
            f"{int(quality_4_6.set_index('feature').loc['occupancy_type', 'minority_n'])} "
            "records."
        ),
        "construction_method": (
            f"Manufactured home has "
            f"{int(quality_4_6.set_index('feature').loc['construction_method', 'minority_n'])} "
            "records."
        ),
        "total_units": (
            f"Smallest raw category has "
            f"{int(quality_4_6.set_index('feature').loc['total_units', 'minority_n'])} "
            f"records; smallest construction-by-units cell has "
            f"{int(joint_scope_4_6['smallest_observed_cell_n'])}."
        ),
    })
)

decision_4_6["representation_status"] = (
    decision_4_6["feature"].map({
        "occupancy_type": "raw_feature_selected",
        "construction_method": "pending_raw_vs_derived",
        "total_units": "pending_raw_vs_derived",
    })
)

decision_4_6["decision"] = (
    decision_4_6["feature"].map({
        "occupancy_type": "keep",
        "construction_method": (
            "keep_for_representation_comparison"
        ),
        "total_units": (
            "keep_for_representation_comparison"
        ),
    })
)

decision_4_6["decision_reason"] = (
    decision_4_6["feature"].map({
        "occupancy_type": (
            "Clean domain, adequate category support and a distinct "
            "occupancy concept."
        ),
        "construction_method": (
            "Passes the quality gate but participates in the "
            "derived dwelling representation."
        ),
        "total_units": (
            "Passes the quality gate as a categorical feature but "
            "must be compared with the derived dwelling representation."
        ),
    })
)

decision_4_6["gsv_player"] = (
    decision_4_6["feature"].map({
        "occupancy_type": "occupancy_type",
        "construction_method": pd.NA,
        "total_units": pd.NA,
    })
)

decision_4_6

,feature,valid_pct,raw_unique_count,dominant_pct,minority_n,minority_pct,business_reason,timing_status,quality_evidence,sparsity_evidence,representation_status,decision,decision_reason,gsv_player
0,occupancy_type,100.0,3,87.77,7100,2.4207,Represents how the property will be occupied.,available_before_decision,100.00% valid; no blank or invalid code.,Smallest category has 7100 records.,raw_feature_selected,keep,"Clean domain, adequate category support and a ...",occupancy_type
1,construction_method,100.0,2,97.68,6815,2.3236,Distinguishes site-built from manufactured hou...,available_before_decision,100.00% valid; no blank or invalid code.,Manufactured home has 6815 records.,pending_raw_vs_derived,keep_for_representation_comparison,Passes the quality gate but participates in th...,<NA>
2,total_units,100.0,9,88.57,108,0.0368,Represents dwelling-unit count as HMDA categor...,available_before_decision,100.00% valid; no blank or invalid code.,Smallest raw category has 108 records; smalles...,pending_raw_vs_derived,keep_for_representation_comparison,Passes the quality gate as a categorical featu...,<NA>


#### Kết luận nhóm 4.6

Cả `occupancy_type`, `construction_method` và `total_units` đều có valid-code coverage 100%, không xuất hiện blank, missing hoặc mã ngoài schema. `occupancy_type` có ba category: principal residence 257.433 hồ sơ (87,77%), investment property 28.768 (9,81%) và second residence 7.100 (2,42%). `construction_method` có 286.486 site-built (97,68%) và 6.815 manufactured home (2,32%).

`total_units` có đủ chín category HMDA và được giữ dưới dạng categorical values, không được parse thành continuous numeric. Category `1` chiếm 259.784 hồ sơ (88,57%); category nhỏ nhất là `100-149` với 108 hồ sơ (0,04%). Các category `25-49`, `50-99`, `100-149` và `>149` có support thấp hơn nhưng chưa bị tự động gộp hoặc loại ở feature selection.

Bảng chéo `construction_method` × `total_units` có đủ 18 combination với tổng số dòng khớp cohort. Tuy nhiên, smallest observed cell chỉ có 2 hồ sơ, nên sự hiện diện của đủ combination không được diễn giải là mọi combination đều có support ổn định. Đây là lý do không chạy association sâu hoặc tự động tạo interaction trong 4.6.

Quyết định hiện tại: KEEP `occupancy_type` như một raw feature và GSV/SL player riêng. `construction_method` và `total_units` đều vượt qua data-quality gate nhưng được đánh dấu KEEP FOR REPRESENTATION COMPARISON; final model representation và GSV/SL player chỉ được khóa sau khi so sánh hai raw features với `derived_dwelling_category` ở mục 6/9. Không đưa đồng thời raw và derived representation vào main model.